In [0]:
%sql
CREATE OR REPLACE TABLE cvm.gold.indicadores_anuais
COMMENT 'Uma linha por empresa x ano com os principais indicadores financeiros'
AS
WITH contas AS (
  -- "pivot": transforma linhas de contas em colunas (como um SOMASES por conta)
  SELECT
    cnpj, ano, max(empresa) AS empresa,
    sum(CASE WHEN demonstracao = 'DRE' AND cd_conta = '3.01' THEN valor_reais END) AS receita_liquida,
    sum(CASE WHEN demonstracao = 'DRE' AND cd_conta = '3.03' THEN valor_reais END) AS lucro_bruto,
    sum(CASE WHEN demonstracao = 'DRE' AND cd_conta = '3.05' THEN valor_reais END) AS ebit,
    sum(CASE WHEN demonstracao = 'DRE' AND cd_conta = '3.11' THEN valor_reais END) AS lucro_liquido,
    sum(CASE WHEN demonstracao = 'BPA' AND cd_conta = '1'       THEN valor_reais END) AS ativo_total,
    sum(CASE WHEN demonstracao = 'BPA' AND cd_conta = '1.01.01' THEN valor_reais END) AS caixa,
    sum(CASE WHEN demonstracao = 'BPA' AND cd_conta = '1.01.02' THEN valor_reais END) AS aplicacoes_cp,
    sum(CASE WHEN demonstracao = 'BPP' AND cd_conta = '2.03'    THEN valor_reais END) AS patrimonio_liquido,
    sum(CASE WHEN demonstracao = 'BPP' AND cd_conta IN ('2.01.04', '2.02.01') THEN valor_reais END) AS divida_bruta,
    -- Depreciação/amortização NÃO está na DRE: vem dos ajustes da DFC (6.01.01.xx).
    -- Como essas contas não são padronizadas, buscamos pela descrição.
    sum(CASE WHEN demonstracao = 'DFC' AND cd_conta LIKE '6.01.01.%'
              AND (lower(ds_conta) LIKE '%deprecia%' OR lower(ds_conta) LIKE '%amortiza%'
                   OR lower(ds_conta) LIKE '%exaust%')
             THEN valor_reais END) AS depreciacao_amortizacao
  FROM cvm.silver.demonstracoes
  GROUP BY cnpj, ano
)
SELECT
  c.cnpj, c.empresa, e.setor, c.ano,
  -- bancos e seguradoras têm DRE diferente; margens não são comparáveis com as demais
  e.setor RLIKE '(?i)banco|intermedia|segur|arrendamento' AS eh_financeira,
  c.receita_liquida, c.lucro_bruto, c.ebit,
  c.ebit + c.depreciacao_amortizacao                                  AS ebitda,
  c.lucro_liquido, c.ativo_total, c.patrimonio_liquido, c.divida_bruta,
  c.divida_bruta - coalesce(c.caixa, 0) - coalesce(c.aplicacoes_cp, 0) AS divida_liquida,
  -- indicadores (try_divide evita erro de divisão por zero)
  round(try_divide(c.lucro_bruto, c.receita_liquida) * 100, 2)                    AS margem_bruta_pct,
  round(try_divide(c.ebit, c.receita_liquida) * 100, 2)                           AS margem_ebit_pct,
  round(try_divide(c.ebit + c.depreciacao_amortizacao, c.receita_liquida) * 100, 2) AS margem_ebitda_pct,
  round(try_divide(c.lucro_liquido, c.receita_liquida) * 100, 2)                  AS margem_liquida_pct,
  round(try_divide(c.lucro_liquido, c.patrimonio_liquido) * 100, 2)               AS roe_pct,
  round(try_divide(c.divida_bruta - coalesce(c.caixa, 0) - coalesce(c.aplicacoes_cp, 0),
                   c.ebit + c.depreciacao_amortizacao), 2)                        AS divida_liq_ebitda,
  round((try_divide(c.receita_liquida,
         lag(c.receita_liquida) OVER (PARTITION BY c.cnpj ORDER BY c.ano)) - 1) * 100, 2)
                                                                                   AS crescimento_receita_pct
FROM contas c
LEFT JOIN cvm.silver.empresas e ON e.cnpj = c.cnpj

In [0]:
%sql
-- Maiores margens EBITDA de 2024 entre empresas não financeiras com receita acima de R$ 1 bi
SELECT empresa, setor,
       format_number(receita_liquida / 1e9, 1) AS receita_bi,
       margem_ebitda_pct, roe_pct, divida_liq_ebitda, crescimento_receita_pct
FROM cvm.gold.indicadores_anuais
WHERE ano = 2024 AND NOT eh_financeira AND receita_liquida > 1e9
ORDER BY margem_ebitda_pct DESC
LIMIT 15

In [0]:
%sql
SELECT ano, format_number(receita_liquida / 1e9, 1) AS receita_bi,
       margem_ebitda_pct, margem_liquida_pct, roe_pct, crescimento_receita_pct
FROM cvm.gold.indicadores_anuais
WHERE empresa LIKE '%AMBEV%'
ORDER BY ano